# Tarea E — Regularización Ridge y validación cruzada sin tocar la prueba

## Parte 1 — Datos

In [1]:
# Subtarea T1: Cargar sklearn.datasets.load_diabetes; reportar número de ejemplos y de atributos; dividir en entrenamiento (75%) y prueba (25%) con train_test_split(
# -*- coding: utf-8 -*-
"""
Subtarea T1 — Tarea E (Ridge y validación cruzada)
===================================================
Parte 1 — Datos:
  1. Cargar sklearn.datasets.load_diabetes (incluido en scikit-learn, sin descargas).
  2. Reportar número de ejemplos y de atributos.
  3. Dividir en entrenamiento (75 %) y prueba (25 %) con train_test_split(random_state=7).
     Esta es la ÚNICA división de la tarea; el conjunto de prueba se reserva para la Parte 3.

Figura: no se requiere.
Salidas: impresión de las cifras principales (4 decimales donde aplica) y
         guardado de todas las cifras en "resultados.json".
"""

import json

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Parámetros fijados por el enunciado (no se toca el conjunto de prueba aquí)
RANDOM_STATE = 7
TEST_SIZE = 0.25

# ---------------------------------------------------------------------------
# 1) Carga del conjunto Diabetes
# ---------------------------------------------------------------------------
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

n_ejemplos_total, n_atributos_total = X.shape

# ---------------------------------------------------------------------------
# 2) Única división de la tarea: 75 % entrenamiento / 25 % prueba
#    (shuffle por defecto de train_test_split, random_state=7)
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

n_ejemplos_entrenamiento = int(X_train.shape[0])
n_ejemplos_prueba = int(X_test.shape[0])

proporcion_entrenamiento = n_ejemplos_entrenamiento / n_ejemplos_total
proporcion_prueba = n_ejemplos_prueba / n_ejemplos_total

# Comprobación de consistencia (sin usar el contenido de los datos)
assert n_ejemplos_entrenamiento + n_ejemplos_prueba == n_ejemplos_total
assert X_train.shape[1] == n_atributos_total == X_test.shape[1]

# ---------------------------------------------------------------------------
# 3) Guardar TODAS las cifras calculadas en resultados.json
#    (se conservan resultados de subtareas previas si el archivo ya existe)
# ---------------------------------------------------------------------------
resultados = {}
try:
    with open("resultados.json", "r", encoding="utf-8") as f:
        previo = json.load(f)
    if isinstance(previo, dict):
        resultados = previo
except (FileNotFoundError, json.JSONDecodeError, ValueError):
    resultados = {}

resultados.update(
    {
        "t1_n_ejemplos_total": float(n_ejemplos_total),
        "t1_n_atributos_total": float(n_atributos_total),
        "t1_n_ejemplos_entrenamiento": float(n_ejemplos_entrenamiento),
        "t1_n_ejemplos_prueba": float(n_ejemplos_prueba),
        "t1_proporcion_entrenamiento": float(proporcion_entrenamiento),
        "t1_proporcion_prueba": float(proporcion_prueba),
        "t1_random_state": float(RANDOM_STATE),
        "t1_test_size": float(TEST_SIZE),
    }
)

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------------
# 4) Reporte de las cifras principales
# ---------------------------------------------------------------------------
print("=== Subtarea T1: carga de datos y única división entrenamiento/prueba ===")
print("Conjunto: sklearn.datasets.load_diabetes")
print(f"Numero de ejemplos (total): {n_ejemplos_total}")
print(f"Numero de atributos (total): {n_atributos_total}")
print(f"Entrenamiento (75%): {n_ejemplos_entrenamiento} ejemplos "
      f"({proporcion_entrenamiento:.4f} del total)")
print(f"Prueba (25%): {n_ejemplos_prueba} ejemplos "
      f"({proporcion_prueba:.4f} del total)")
print(f"Division: train_test_split(test_size={TEST_SIZE}, random_state={RANDOM_STATE})")
print("El conjunto de prueba queda reservado; solo se usara una vez, al final de la Parte 3.")
print("Cifras guardadas en resultados.json")


=== Subtarea T1: carga de datos y única división entrenamiento/prueba ===
Conjunto: sklearn.datasets.load_diabetes
Numero de ejemplos (total): 442
Numero de atributos (total): 10
Entrenamiento (75%): 331 ejemplos (0.7489 del total)
Prueba (25%): 111 ejemplos (0.2511 del total)
Division: train_test_split(test_size=0.25, random_state=7)
El conjunto de prueba queda reservado; solo se usara una vez, al final de la Parte 3.
Cifras guardadas en resultados.json


Uso el conjunto Diabetes que trae incluido scikit-learn (`load_diabetes`), sin descargar ningún otro conjunto. Tiene 442 ejemplos y 10 atributos. Hago la única división de la tarea con `train_test_split(test_size=0.25, random_state=7)`, que deja 331 ejemplos de entrenamiento y 111 de prueba, la proporción 75/25 pedida. La prueba queda reservada y no la toco en esta parte: se usará una sola vez, al final de la Parte 3.

## Parte 2 — Selección de alpha por validación cruzada

In [2]:
# Subtarea T2: Construir pipeline StandardScaler + Ridge; GridSearchCV sobre los 13 valores de numpy.logspace(-3, 3, 13) con KFold(n_splits=5, shuffle=True, random_s
# -*- coding: utf-8 -*-
"""
Subtarea T2 — Pipeline StandardScaler + Ridge con selección de alpha por
validación cruzada (GridSearchCV), usando SOLO el conjunto de entrenamiento.

Datos: sklearn.datasets.load_diabetes
División: train_test_split(test_size=0.25, random_state=7)  (misma que T1)
Grid:     13 valores de numpy.logspace(-3, 3, 13)
CV:       KFold(n_splits=5, shuffle=True, random_state=7), scoring='r2'
Figura:   no

Nota: en esta subtarea NO se predice sobre ningún conjunto. El único R2
reportado es el medio de validación cruzada calculado por GridSearchCV
dentro del conjunto de entrenamiento. El conjunto de prueba queda reservado.
"""

import json
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


# ----------------------------------------------------------------------
# 1) Datos y única división entrenamiento/prueba (idéntica a la T1)
# ----------------------------------------------------------------------
data = load_diabetes()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# El conjunto de prueba NO se usa en esta subtarea (ni para ajustar el
# escalador, ni para elegir alpha). Solo se entrena con X_train / y_train.


# ----------------------------------------------------------------------
# 2) Pipeline StandardScaler + Ridge
# ----------------------------------------------------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

# 13 valores de alpha en escala logarítmica de 10^-3 a 10^3
alphas = np.logspace(-3, 3, 13)
param_grid = {'ridge__alpha': alphas}


# ----------------------------------------------------------------------
# 3) GridSearchCV con KFold(5, shuffle=True, random_state=7) y scoring='r2'
#    Ajustado únicamente con el conjunto de entrenamiento. El R2 medio de
#    validación cruzada lo calcula GridSearchCV internamente: en cada fold
#    el modelo se ajusta con 4 folds y se evalúa con el fold restante, por
#    lo que nunca se predice sobre los mismos datos con los que se ajustó.
# ----------------------------------------------------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=7)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='r2',
    cv=cv
)
grid.fit(X_train, y_train)

best_alpha = float(grid.best_params_['ridge__alpha'])
best_cv_r2 = float(grid.best_score_)
best_index = int(grid.best_index_)

# Resultados completos de la malla (media y desviación del R2 de CV por alpha)
mean_scores = np.asarray(grid.cv_results_['mean_test_score'], dtype=float)
std_scores = np.asarray(grid.cv_results_['std_test_score'], dtype=float)


# ----------------------------------------------------------------------
# 4) Guardar TODAS las cifras en resultados.json (conservando claves previas)
# ----------------------------------------------------------------------
nuevas_cifras = {
    't2_n_alphas_evaluados': int(len(alphas)),
    't2_alphas_evaluados': [float(a) for a in alphas],
    't2_r2_medio_cv_por_alpha': [float(s) for s in mean_scores],
    't2_r2_std_cv_por_alpha': [float(s) for s in std_scores],
    't2_alpha_elegido': best_alpha,
    't2_r2_medio_cv': best_cv_r2,
    't2_indice_mejor_alpha': best_index,
    't2_n_splits_cv': 5,
    't2_shuffle_cv': True,
    't2_random_state_cv': 7,
    't2_scoring': 'r2',
    't2_n_ejemplos_entrenamiento': int(X_train.shape[0]),
    't2_n_ejemplos_prueba_reservados': int(X_test.shape[0]),
}

try:
    with open('resultados.json', 'r', encoding='utf-8') as f:
        resultados = json.load(f)
    if not isinstance(resultados, dict):
        resultados = {}
except Exception:
    resultados = {}

resultados.update(nuevas_cifras)

with open('resultados.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)


# ----------------------------------------------------------------------
# 5) Salida por consola (cifras principales con 4 decimales)
# ----------------------------------------------------------------------
print("=== Subtarea T2: pipeline StandardScaler + Ridge, alpha por validacion cruzada ===")
print("Datos: sklearn.datasets.load_diabetes")
print(f"Entrenamiento usado para el grid: {X_train.shape[0]} ejemplos "
      f"(prueba reservada: {X_test.shape[0]} ejemplos, sin usar aqui)")
print("Pipeline: StandardScaler -> Ridge")
print("Busqueda: GridSearchCV sobre 13 valores de np.logspace(-3, 3, 13)")
print("CV: KFold(n_splits=5, shuffle=True, random_state=7), scoring='r2'")
print("")
print("R2 medio de CV por alpha:")
for a, m, s in zip(alphas, mean_scores, std_scores):
    marca = "  <-- mejor" if float(a) == best_alpha else ""
    print(f"  alpha={float(a):10.4f} : R2 medio CV = {m:.4f} (+/- {s:.4f}){marca}")
print("")
print(f"Alpha elegido: {best_alpha:.4f}")
print(f"R2 medio de validacion cruzada (5 folds): {best_cv_r2:.4f}")
print("Cifras guardadas en resultados.json")


=== Subtarea T2: pipeline StandardScaler + Ridge, alpha por validacion cruzada ===
Datos: sklearn.datasets.load_diabetes
Entrenamiento usado para el grid: 331 ejemplos (prueba reservada: 111 ejemplos, sin usar aqui)
Pipeline: StandardScaler -> Ridge
Busqueda: GridSearchCV sobre 13 valores de np.logspace(-3, 3, 13)
CV: KFold(n_splits=5, shuffle=True, random_state=7), scoring='r2'

R2 medio de CV por alpha:
  alpha=    0.0010 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0032 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0100 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0316 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.1000 : R2 medio CV = 0.4724 (+/- 0.0705)
  alpha=    0.3162 : R2 medio CV = 0.4724 (+/- 0.0709)  <-- mejor
  alpha=    1.0000 : R2 medio CV = 0.4722 (+/- 0.0717)
  alpha=    3.1623 : R2 medio CV = 0.4714 (+/- 0.0730)
  alpha=   10.0000 : R2 medio CV = 0.4708 (+/- 0.0739)
  alpha=   31.6228 : R2 medio CV = 0.4698 (+/- 0.0741)
  alpha=  100.0000 : R2 medio 

Construyo un pipeline con `StandardScaler` seguido de `Ridge` y elijo alpha con `GridSearchCV` sobre los 13 valores de `np.logspace(-3, 3, 13)`, usando `KFold(n_splits=5, shuffle=True, random_state=7)` y `scoring="r2"`. Todo el procedimiento corre solo con los 331 ejemplos de entrenamiento. El resultado es alpha = 0.3162, con un R2 medio de validación cruzada de 0.4724 (desviación entre folds de 0.0709). La celda de código imprime el R2 medio de los 13 alphas: la curva es casi plana para alphas pequeños, baja suavemente desde alpha = 1 y cae con fuerza a partir de alpha = 100.

## Parte 3 — Evaluación en prueba

In [3]:
# Subtarea T3: Con el alpha elegido en T2, evaluar sobre el conjunto de prueba (usado una sola vez): R2 y RMSE del Ridge; comparar con LinearRegression con el mismo 
# -*- coding: utf-8 -*-
"""
Subtarea T3: con el alpha elegido en T2, evaluar sobre el conjunto de prueba
(usado una sola vez) el R2 y el RMSE del Ridge, y comparar con LinearRegression
con el mismo escalado (StandardScaler) evaluada sobre la misma prueba.

El script es independiente: recalcula el alpha de T2 con exactamente el mismo
procedimiento (GridSearchCV, KFold(n_splits=5, shuffle=True, random_state=7),
scoring="r2", alphas = np.logspace(-3, 3, 13)) usando SOLO el conjunto de
entrenamiento. El conjunto de prueba se usa una única vez, al final.
"""

import json
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

RUTA_RESULTADOS = "resultados.json"


def main():
    # ------------------------------------------------------------------
    # 1) Datos y única división entrenamiento/prueba (idéntica a T1)
    # ------------------------------------------------------------------
    datos = load_diabetes()
    X, y = datos.data, datos.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=7
    )

    # ------------------------------------------------------------------
    # 2) Re-cálculo del alpha elegido en T2 (solo con el entrenamiento)
    # ------------------------------------------------------------------
    alphas = np.logspace(-3, 3, 13)

    pipe_ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge()),
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=7)
    busqueda = GridSearchCV(
        estimator=pipe_ridge,
        param_grid={"ridge__alpha": alphas},
        cv=cv,
        scoring="r2",
    )
    # La prueba NO participa en esta búsqueda
    busqueda.fit(X_train, y_train)

    alpha_elegido = float(busqueda.best_params_["ridge__alpha"])
    r2_cv_alpha_elegido = float(busqueda.best_score_)
    r2_medio_cv_por_alpha = [float(v) for v in busqueda.cv_results_["mean_test_score"]]
    r2_std_cv_por_alpha = [float(v) for v in busqueda.cv_results_["std_test_score"]]

    # ------------------------------------------------------------------
    # 3) Evaluación ÚNICA en prueba: Ridge con el alpha elegido
    #    (escalador ajustado solo con entrenamiento, dentro del pipeline)
    # ------------------------------------------------------------------
    ridge_final = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha_elegido)),
    ])
    ridge_final.fit(X_train, y_train)
    y_pred_ridge = ridge_final.predict(X_test)
    ridge_r2_prueba = float(r2_score(y_test, y_pred_ridge))
    ridge_rmse_prueba = float(np.sqrt(mean_squared_error(y_test, y_pred_ridge)))

    # ------------------------------------------------------------------
    # 4) Comparación: LinearRegression con el mismo escalado,
    #    evaluada sobre la misma prueba
    # ------------------------------------------------------------------
    linreg_final = Pipeline([
        ("scaler", StandardScaler()),
        ("linreg", LinearRegression()),
    ])
    linreg_final.fit(X_train, y_train)
    y_pred_linreg = linreg_final.predict(X_test)
    linreg_r2_prueba = float(r2_score(y_test, y_pred_linreg))
    linreg_rmse_prueba = float(np.sqrt(mean_squared_error(y_test, y_pred_linreg)))

    # Diferencias (positivas => Ridge mejor en prueba)
    diff_r2 = float(ridge_r2_prueba - linreg_r2_prueba)
    diff_rmse = float(linreg_rmse_prueba - ridge_rmse_prueba)
    if diff_r2 > 0:
        mejor_r2 = "ridge"
    elif diff_r2 < 0:
        mejor_r2 = "linear_regression"
    else:
        mejor_r2 = "empate"

    # ------------------------------------------------------------------
    # 5) Guardar TODAS las cifras calculadas en resultados.json
    # ------------------------------------------------------------------
    resultados = {
        "t3_n_ejemplos_entrenamiento": float(X_train.shape[0]),
        "t3_n_ejemplos_prueba": float(X_test.shape[0]),
        "t3_n_alphas_evaluados": float(len(alphas)),
        "t3_alphas_evaluados": [float(a) for a in alphas],
        "t3_r2_medio_cv_por_alpha": r2_medio_cv_por_alpha,
        "t3_r2_std_cv_por_alpha": r2_std_cv_por_alpha,
        "t3_alpha_elegido": alpha_elegido,
        "t3_r2_medio_cv_alpha_elegido": r2_cv_alpha_elegido,
        "t3_ridge_r2_prueba": ridge_r2_prueba,
        "t3_ridge_rmse_prueba": ridge_rmse_prueba,
        "t3_linreg_r2_prueba": linreg_r2_prueba,
        "t3_linreg_rmse_prueba": linreg_rmse_prueba,
        "t3_diferencia_r2_ridge_menos_lineal": diff_r2,
        "t3_diferencia_rmse_lineal_menos_ridge": diff_rmse,
        "t3_modelo_con_mejor_r2_prueba": mejor_r2,
    }

    with open(RUTA_RESULTADOS, "w") as f:
        json.dump(resultados, f, indent=2)

    # ------------------------------------------------------------------
    # 6) Salida por pantalla (cifras principales con 4 decimales)
    # ------------------------------------------------------------------
    print("=== Subtarea T3: evaluacion unica en el conjunto de prueba ===")
    print("Datos: sklearn.datasets.load_diabetes")
    print(
        "Entrenamiento: %d ejemplos | Prueba: %d ejemplos (usada una sola vez, aqui)"
        % (X_train.shape[0], X_test.shape[0])
    )
    print("Alpha re-elegido con el mismo procedimiento de T2 (solo entrenamiento):")
    print(
        "  alpha = %.4f | R2 medio de CV (5 folds) = %.4f"
        % (alpha_elegido, r2_cv_alpha_elegido)
    )
    print()
    print("Resultados sobre el conjunto de prueba:")
    print(
        "  Ridge (alpha=%.4f):                R2 = %.4f | RMSE = %.4f"
        % (alpha_elegido, ridge_r2_prueba, ridge_rmse_prueba)
    )
    print(
        "  LinearRegression (mismo escalado): R2 = %.4f | RMSE = %.4f"
        % (linreg_r2_prueba, linreg_rmse_prueba)
    )
    print()
    print("Diferencia R2   (Ridge - Lineal):   %.4f" % diff_r2)
    print("Diferencia RMSE (Lineal - Ridge):   %.4f" % diff_rmse)
    print("Mejor R2 en prueba: %s" % mejor_r2)
    print("Cifras guardadas en resultados.json")


if __name__ == "__main__":
    main()


=== Subtarea T3: evaluacion unica en el conjunto de prueba ===
Datos: sklearn.datasets.load_diabetes
Entrenamiento: 331 ejemplos | Prueba: 111 ejemplos (usada una sola vez, aqui)
Alpha re-elegido con el mismo procedimiento de T2 (solo entrenamiento):
  alpha = 0.3162 | R2 medio de CV (5 folds) = 0.4724

Resultados sobre el conjunto de prueba:
  Ridge (alpha=0.3162):                R2 = 0.5037 | RMSE = 54.3838
  LinearRegression (mismo escalado): R2 = 0.5028 | RMSE = 54.4331

Diferencia R2   (Ridge - Lineal):   0.0009
Diferencia RMSE (Lineal - Ridge):   0.0493
Mejor R2 en prueba: ridge
Cifras guardadas en resultados.json


Con el alpha ya fijado en 0.3162, reentreno el pipeline con todo el conjunto de entrenamiento y evalúo una sola vez sobre la prueba: Ridge obtiene R2 = 0.5037 y RMSE = 54.3838. Sobre la misma prueba, la regresión lineal sin regularizar (con el mismo escalado) obtiene R2 = 0.5028 y RMSE = 54.4331. Las diferencias son mínimas (0.0009 en R2 y 0.0493 en RMSE a favor de Ridge), lo cual es coherente con la curva de CV casi plana en alphas pequeños: en estos datos, la regularización apenas cambia el modelo lineal.

## Parte 4 — Figura

In [4]:
# Subtarea T4: Dibujar con matplotlib el R2 medio de validación cruzada (de T2) contra alpha, con el eje de alpha en escala logarítmica (semilogx); la figura queda v
# === Subtarea T4: figura del R2 medio de validación cruzada (de T2) contra alpha ===
# Recalcula de forma independiente la búsqueda de T2 (mismos datos, partición,
# pipeline, grid de alphas y esquema de CV) y dibuja el R2 medio de CV frente a
# alpha con el eje de alpha en escala logarítmica (semilogx).
# La figura se guarda como PNG y además queda visible como salida de la celda.

import json

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------
# 1) Datos y partición (idénticos a T2: 331 entrenamiento / 111 prueba)
#    El conjunto de prueba NO se usa aquí para nada.
# ------------------------------------------------------------------
datos = load_diabetes()
X, y = datos.data, datos.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, shuffle=True
)

# ------------------------------------------------------------------
# 2) Misma búsqueda que T2: StandardScaler -> Ridge, grid de 13 alphas,
#    KFold(5, shuffle=True, random_state=7), scoring='r2'
# ------------------------------------------------------------------
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])

alphas = np.logspace(-3, 3, 13)
param_grid = {"ridge__alpha": alphas}

cv = KFold(n_splits=5, shuffle=True, random_state=7)
busqueda = GridSearchCV(pipeline, param_grid, cv=cv, scoring="r2")
busqueda.fit(X_train, y_train)

alphas_eval = np.array(
    busqueda.cv_results_["param_ridge__alpha"], dtype=float
)
r2_medio = np.array(busqueda.cv_results_["mean_test_score"], dtype=float)
r2_std = np.array(busqueda.cv_results_["std_test_score"], dtype=float)

indice_mejor = int(np.argmax(r2_medio))
mejor_alpha = float(alphas_eval[indice_mejor])
mejor_r2 = float(r2_medio[indice_mejor])

# ------------------------------------------------------------------
# 3) Figura: R2 medio de CV contra alpha, eje x en escala logarítmica
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(
    alphas_eval,
    r2_medio,
    marker="o",
    color="tab:blue",
    linewidth=1.5,
    label="R2 medio de CV (5 folds)",
)
ax.axvline(
    mejor_alpha,
    color="tab:red",
    linestyle="--",
    linewidth=1.2,
    label=f"Mejor alpha = {mejor_alpha:.4g}",
)
ax.set_xlabel("alpha (escala logarítmica)")
ax.set_ylabel("R2 medio de validación cruzada")
ax.set_title("Ridge en diabetes: R2 medio de CV frente a alpha")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="lower left")
fig.tight_layout()

# Guardar el PNG y mostrar la figura como salida de la celda del notebook
fig.savefig("t4_r2_vs_alpha.png", dpi=150)
display(fig)
plt.close(fig)

# ------------------------------------------------------------------
# 4) Guardar TODAS las cifras calculadas en resultados.json
#    (se conservan las claves de subtareas anteriores si el archivo existe)
# ------------------------------------------------------------------
try:
    with open("resultados.json", "r", encoding="utf-8") as f:
        resultados = json.load(f)
    if not isinstance(resultados, dict):
        resultados = {}
except (FileNotFoundError, json.JSONDecodeError):
    resultados = {}

resultados["t4_n_alphas_evaluados"] = int(len(alphas_eval))
resultados["t4_alphas_evaluados"] = [float(a) for a in alphas_eval]
resultados["t4_r2_medio_cv_por_alpha"] = [float(v) for v in r2_medio]
resultados["t4_r2_std_cv_por_alpha"] = [float(v) for v in r2_std]
resultados["t4_indice_mejor_alpha"] = indice_mejor
resultados["t4_alpha_elegido"] = mejor_alpha
resultados["t4_r2_medio_cv_mejor_alpha"] = mejor_r2
resultados["t4_n_splits_cv"] = 5
resultados["t4_shuffle_cv"] = True
resultados["t4_random_state_cv"] = 7
resultados["t4_scoring"] = "r2"
resultados["t4_n_ejemplos_entrenamiento"] = int(X_train.shape[0])
resultados["t4_n_ejemplos_prueba_reservados"] = int(X_test.shape[0])
resultados["t4_figura"] = "t4_r2_vs_alpha.png"

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------------
# 5) Impresión de las cifras principales con 4 decimales
# ------------------------------------------------------------------
print("=== Subtarea T4: figura R2 medio de CV frente a alpha ===")
print(f"Alphas evaluados: {len(alphas_eval)} (np.logspace(-3, 3, 13))")
print("R2 medio de validación cruzada por alpha:")
for a, m, s in zip(alphas_eval, r2_medio, r2_std):
    marca = "  <-- mejor" if float(a) == mejor_alpha else ""
    print(f"  alpha={a:10.4f} : R2 medio CV = {m:.4f} (+/- {s:.4f}){marca}")
print()
print(f"Mejor alpha: {mejor_alpha:.4f}")
print(f"R2 medio de CV en el mejor alpha: {mejor_r2:.4f}")
print("Figura guardada como t4_r2_vs_alpha.png y mostrada arriba")
print("Cifras guardadas en resultados.json")


<Figure size 800x500 with 1 Axes>

=== Subtarea T4: figura R2 medio de CV frente a alpha ===
Alphas evaluados: 13 (np.logspace(-3, 3, 13))
R2 medio de validación cruzada por alpha:
  alpha=    0.0010 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0032 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0100 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.0316 : R2 medio CV = 0.4724 (+/- 0.0703)
  alpha=    0.1000 : R2 medio CV = 0.4724 (+/- 0.0705)
  alpha=    0.3162 : R2 medio CV = 0.4724 (+/- 0.0709)  <-- mejor
  alpha=    1.0000 : R2 medio CV = 0.4722 (+/- 0.0717)
  alpha=    3.1623 : R2 medio CV = 0.4714 (+/- 0.0730)
  alpha=   10.0000 : R2 medio CV = 0.4708 (+/- 0.0739)
  alpha=   31.6228 : R2 medio CV = 0.4698 (+/- 0.0741)
  alpha=  100.0000 : R2 medio CV = 0.4570 (+/- 0.0732)
  alpha=  316.2278 : R2 medio CV = 0.3995 (+/- 0.0683)
  alpha= 1000.0000 : R2 medio CV = 0.2790 (+/- 0.0514)

Mejor alpha: 0.3162
R2 medio de CV en el mejor alpha: 0.4724
Figura guardada como t4_r2_vs_alpha.png y mostrada arriba
Cifras g

La figura muestra el R2 medio de validación cruzada frente a alpha, con el eje de alpha en escala logarítmica. Se distinguen tres zonas: una meseta para alphas pequeños, donde el R2 medio se mantiene entre 0.4698 (alpha = 31.6228) y 0.4724 (alpha = 0.3162); el máximo en alpha = 0.3162; y una caída fuerte hacia alpha = 1000, donde el R2 baja a 0.2790 porque la penalización ya restringe demasiado los coeficientes.

![R2 medio de validación cruzada frente a alpha (eje de alpha en escala logarítmica), con el máximo en alpha = 0.3162](T4_t4_r2_vs_alpha.png)

## Parte 5 — Pregunta conceptual

**Por qué elegir alpha mirando el R2 de prueba daría una estimación optimista.** El conjunto de prueba solo estima el desempeño con datos nuevos si no participa en ninguna decisión del modelado. Si elijo alpha maximizando el R2 de prueba, la prueba deja de ser datos no vistos: el hiperparámetro se adapta a los 111 puntos concretos de esa muestra, incluido su ruido, y parte de lo que parece buen desempeño es señal de la muestra de prueba y no capacidad de generalizar. Además, al comparar 13 alphas y quedarme con el mejor sobre la prueba, estaría reportando el máximo de 13 estimaciones ruidosas; aunque los 13 modelos fueran igual de buenos, ese máximo tiende a quedar por encima del nivel real del modelo elegido (sesgo de selección), y con una prueba de solo 111 ejemplos la varianza es alta, así que el exceso puede ser grande. También quedaría la prueba "gastada": después de usarla para elegir alpha, no me quedaría ningún dato limpio para confirmar el desempeño. Por eso el orden correcto es el que seguí: alpha se elige con validación cruzada dentro del entrenamiento, y la prueba se usa una sola vez, al final.

**Diferencia entre el R2 de validación cruzada y el de prueba en mis resultados.** El R2 medio de CV del alpha elegido (0.3162) fue 0.4724, mientras que el R2 de prueba fue 0.5037: el de prueba queda por encima. Eso no es una contradicción ni una fuga de datos, y tiene explicaciones naturales. Primero, se calculan sobre datos distintos: la CV evalúa sobre folds del conjunto de entrenamiento, mientras que la prueba son 111 ejemplos que ningún modelo vio. Segundo, cada modelo de CV se entrena con cuatro quintos del entrenamiento, mientras que el modelo final se reentrena con los 331 ejemplos completos, y más datos suele ayudar un poco. Tercero, la CV tiene varianza propia: la desviación estándar del R2 entre folds fue 0.0709, y la diferencia observada entre 0.5037 y 0.4724 es menor que esa variabilidad entre folds, así que cabe de sobra dentro del comportamiento normal del estimador. (El R2 de CV del alpha elegido también arrastra un pequeño sesgo de selección por ser el mejor de 13 valores, pero ese sesgo se queda dentro del entrenamiento y no compromete la limpieza de la prueba.) En resumen: el R2 de CV (0.4724) fue solo el criterio para elegir alpha, y el R2 de prueba (0.5037) es la estimación definitiva de generalización, obtenida con un único uso del conjunto de prueba. Si hubiera elegido alpha mirando la prueba, el número reportado habría sido el mejor de 13 intentos sobre esos mismos 111 puntos y habría heredado el optimismo descrito, sin que yo pudiera distinguir cuánto era señal y cuánto suerte de la muestra.